# RFP Parsing P4 - HWPX 250 Mini Pilot

기존 P3 250개와 동일한 문서 샘플을 사용해 HWPX 기반 table-aware corpus를 생성합니다.

목표는 690개 전체 확장 전에 아래를 빠르게 확인하는 것입니다.

```text
1. P3와 같은 250개 문서 샘플 로드
2. HWPX가 있는 HWP 문서는 HWPX ZIP/XML 파싱 우선 적용
3. PDF 문서와 HWPX 변환 실패 문서는 fallback 처리
4. v1 clean text baseline 생성
5. v2 HWPX table-aware structured corpus 생성
6. chunk/source_store 연결, ID 중복, 파일 크기 검증
```

산출물 위치:

```text
outputs/parsing_p4_hwpx_250/
├─ chunks_v1_250.jsonl
├─ chunks_v2_250.jsonl
├─ source_store_v1_250.jsonl
├─ source_store_250.jsonl
├─ metadata_light_250.xlsx
├─ manifest.json
├─ validation_report_v1.json
├─ validation_report.json
├─ table_preview_250.csv
└─ README.md
```

GitHub에는 원본 RFP, source_store, Chroma DB, embedding cache를 올리지 않습니다.

In [2]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

MODULE_REL = Path('src') / 'parsing' / 'rfp_p4_hwpx_corpus.py'


def _root_score(module_path: Path) -> tuple:
    text = str(module_path).replace('\\', '/')
    return (
        0 if 'DB_RAG_Codeit_Project' in text else 1,
        0 if '/src/parsing/rfp_p4_hwpx_corpus.py' in text else 1,
        len(text),
    )


def resolve_project_root() -> Path:
    env_root = os.environ.get('RFP_PROJECT_ROOT')
    candidates = []
    if env_root:
        candidates.append(Path(env_root))

    cwd = Path.cwd()
    candidates.extend([
        cwd,
        *cwd.parents,
        Path('/content/drive/MyDrive/DB_RAG_Codeit_Project'),
        Path('/content/drive/MyDrive/DB_RAG_Codeit_Project/DB_RAG_Codeit_Project'),
        Path('/content/drive/MyDrive/Colab Notebooks/DB_RAG_Codeit_Project'),
        Path(r'C:\Users\yoosy\Desktop\codeit\project_2nd'),
    ])

    seen = set()
    for base in candidates:
        key = str(base)
        if key in seen:
            continue
        seen.add(key)
        if (base / MODULE_REL).exists():
            return base

    search_roots = [
        Path('/content/drive/MyDrive'),
        Path('/content/drive/Shareddrives'),
        Path('/content/drive'),
        cwd,
    ]
    matches = []
    for root in search_roots:
        if root.exists():
            try:
                matches.extend(root.rglob('rfp_p4_hwpx_corpus.py'))
            except Exception as exc:
                print(f'skip search root: {root} | {exc}')

    matches = sorted(set(matches), key=_root_score)
    if matches:
        print('found rfp_p4_hwpx_corpus.py candidates:')
        for i, p in enumerate(matches[:20]):
            print(f'  [{i}] {p}')
        module_path = matches[0]
        return module_path.parents[2]

    raise FileNotFoundError(
        'rfp_p4_hwpx_corpus.py? ?? ?????. '
        'Google Drive? mount ????, src/parsing ??? ?? ?? ??? ????? ?????.'
    )


PROJECT_ROOT = resolve_project_root()
parsing_src = PROJECT_ROOT / 'src' / 'parsing'
if str(parsing_src) not in sys.path:
    sys.path.insert(0, str(parsing_src))

from rfp_p4_hwpx_corpus import write_p4_corpus

print('PROJECT_ROOT:', PROJECT_ROOT)
print('module exists:', (PROJECT_ROOT / MODULE_REL).exists())
print('P3 250 metadata exists:', (PROJECT_ROOT / 'outputs' / 'parsing_p3_250' / 'metadata_light_250.xlsx').exists())
print('HWPX folder exists:', (PROJECT_ROOT / 'data' / 'hwpx_664').exists())


FileNotFoundError: rfp_p4_hwpx_corpus.py? ?? ?????. Google Drive? mount ????, src/parsing ??? ?? ?? ??? ????? ?????.

## 1. Generate P4 HWPX 250 Corpus

예상 시간은 환경과 파일 크기에 따라 달라집니다.

```text
최소: 3~5분
중앙: 8~15분
최대: 20~40분
```

근거: 250개 문서 중 대부분은 HWPX XML 파싱이며, 표가 많은 문서는 `rows/rowspan/colspan` 구조화 때문에 시간이 늘어날 수 있습니다.

In [ ]:
result = write_p4_corpus(PROJECT_ROOT, limit=250)

print('output_dir:', result['output_dir'])
print('\n[v1 validation]')
print(json.dumps(result['report_v1'], ensure_ascii=False, indent=2))
print('\n[v2 validation]')
print(json.dumps(result['report_v2'], ensure_ascii=False, indent=2))

## 2. Quick Check

핵심은 아래 항목이 PASS인지 보는 것입니다.

```text
duplicate_chunk_id_count = 0
duplicate_source_store_id_count = 0
missing_source_store_ref = 0
parse_success_docs = 250
source_format_counts에서 hwpx가 충분히 사용되는지
table_count / merged_cell_count가 기록되는지
```

In [ ]:
summary_df = result['summary_df']
display_cols = [
    'rank_index', 'source_file', 'source_format', 'parser_status',
    'clean_char_len', 'non_table_clean_char_len', 'table_count', 'image_count',
    'chunk_count_v1', 'chunk_count_v2', 'final_budget', 'final_project_duration'
]

print('--- source_format counts ---')
print(summary_df['source_format'].value_counts(dropna=False).to_string())

print('\n--- parser_status counts ---')
print(summary_df['parser_status'].value_counts(dropna=False).to_string())

print('\n--- top 10 document summary ---')
display(summary_df[display_cols].head(10))

## 3. Table Preview

표 구조가 제대로 잡혔는지 사람이 확인하는 셀입니다. `columns_candidate`, `table_shape`, `preview`를 봅니다.

In [ ]:
table_preview_df = result['table_preview_df']
if table_preview_df.empty:
    print('table preview is empty')
else:
    pd.set_option('display.max_colwidth', 240)
    display(table_preview_df.head(10))